In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기
# ============================================================

train = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '2_train_clean.parquet')
val = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '1_val_fe.parquet')

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

In [2]:
# ============================================================
# 1. 라이브러리 불러오기
# ============================================================

import numpy as np
import optuna

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

c:\Users\20162180\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")

<변수개수 검증>
    수위변수: 6
    강우변수: 9, 
    환경변수: 7, 
    수위차분(wl_t포함): 6,
    강우강도(rain_1h포함): 6
    기존 숫자열(수위 + 강우 + 환경): 22, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): 27,
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): 31



In [ ]:
# =============================================================
# 2. train 데이터를 튜닝 검증용으로 재분할(val은 최종 val로 남겨둠)
# =============================================================
# 2-1. train -> train_optuna, val_optuna

# 기존 train의 event_id 확인
print(f"train의 event_id 개수: {train['event_id'].nunique()}")  # 98개

optuna_split_idx = int((train['event_id'].nunique()) * 0.8)
print(f"재분할 train_optuna event_id 개수: {optuna_split_idx}")
print()

train_optuna = train.loc[
    train['event_id'] <= optuna_split_idx
]
val_optuna = train.loc[
    train['event_id'] > optuna_split_idx
]

print(f"전체 train event_id 개수: {train['event_id'].nunique()}")
print(f"train_optuna event_id 개수: {train_optuna['event_id'].nunique()}")
print(f"val_optuna event_id 개수: {val_optuna['event_id'].nunique()}")

train의 event_id 개수: 98
재분할 train_optuna event_id 개수: 78

전체 train event_id 개수: 98
train_optuna event_id 개수: 78
val_optuna event_id 개수: 20


In [ ]:
# =============================================================
# 3. 종속변수 설정
# =============================================================

y_train_optuna = train_optuna[target_col]
y_val_optuna = val_optuna[target_col]

In [ ]:
print(len(train_optuna['station_id'].unique()))
print(len(val_optuna['station_id'].unique()))
print(
    set(train_optuna['station_id'].unique()) - 
    set(val_optuna['station_id'].unique())
    ) # set() 비어있으면 둘은 같은 집합, OHE 문제 없음

465
465
set()


In [ ]:
# ============================================================
# 4. station_id 원핫인코딩 독립변수 생성
# ============================================================

# OHE 전 독립변수 데이터셋
X_train_optuna = train_optuna[num_cols_fe]
X_val_optuna = val_optuna[num_cols_fe]

# 원핫인코딩
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    handle_unknown='ignore',  # train에 없던 범주가 나와도 에러 X
    sparse_output=True  # sparse: 희소한, 희소행렬(0이 아닌 값이 드문 행렬)
    # 원핫인코딩 결과를 전체 다 저장 X, 희소한 1의 값의 위치만 저장(메모리 사용 적음)
)

# 학습은 train만
ohe.fit(train_optuna[['station_id']])  # sklearn 인코더는 2차원 행렬을 받는다.

# 변환
train_st_id_ohe = ohe.transform(train_optuna[['station_id']])
val_st_id_ohe = ohe.transform(val_optuna[['station_id']])
# 여기까지는 일반적인 인코딩 과정

# train, val 셋에 합치기
from scipy.sparse import hstack
# scipy안에 희소행렬을 다루는 서브캐키지 scipy.sparse
# hstack(horizontal stack)은 concat(axis=1)과 같은 역할, df가 아닌 행렬도 붙일 수 있는 함수

X_train_optuna_ohe = hstack([
    X_train_optuna,
    train_st_id_ohe
]).tocsr()  # df + 희소행렬(ohe) -> csr형태로 저장하여 sparse 상태 유지
# 이 변수는 더이상 df가 아님

X_val_optuna_ohe = hstack([
    X_val_optuna,
    val_st_id_ohe
]).tocsr()

print("원핫인코딩 전 후 데이터셋 크기 비교: 행은 같고 열만 늘어남")
print(X_train_optuna.shape, X_train_optuna_ohe.shape)
print(X_val_optuna.shape, X_val_optuna_ohe.shape)
# 제공된 데이터셋의 관측소 개수는 465개(md파일에 명시)
# 기존컬럼 31개 + 465 = 496이 정상

원핫인코딩 전 후 데이터셋 크기 비교: 행은 같고 열만 들어남
(404009, 31) (404009, 496)
(212309, 31) (212309, 496)


In [ ]:
# ============================================================
# 5. optuna 목적함수 정의
# ============================================================

# 이해용 설명
# 최종 옵티마이저 함수의 모습
study.optimize(  # 
    objective,
    n_trials=3
)


def objective(trial): # ob~라는 함수, trail이라는 입력값이 들어가면? 다음과 같이 행한다.

    # 하이퍼파라미터: depth, learning_rate, 12_leaf_reg, random_strength
    depth = trial.suggest_int(
        'depth',
        4,
        9
    )
